# Train YOLOv8 operculum segmentation

Trains `yolov8x-seg` on `datasets/operculum` (1080 px, mild albumentations degradations, geometric and colour augmentation) and validates the best checkpoint. Runs are written to `runs/segment/YOLOv8_Operculum/` next to this notebook.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
from ultralytics import YOLO

In [5]:
%pip install ultralytics
%pip install albumentations

from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()

Ultralytics 8.4.56  Python-3.10.14 torch-2.8.0+cu126 CUDA:0 (NVIDIA RTX 3500 Ada Generation Laptop GPU, 12282MiB)
Setup complete  (24 CPUs, 63.7 GB RAM, 1168.6/1905.5 GB disk)


In [ ]:
custom_transforms = [
    A.OneOf([
        A.MotionBlur(blur_limit=(3, 5)),
        A.GaussianBlur(blur_limit=(3, 3)),
    ], p=0.15),
    A.Downscale(scale_range=(0.6, 0.9), p=0.15),
    A.ImageCompression(quality_range=(60, 95), p=0.2),
    A.GaussNoise(std_range=(0.01, 0.03), p=0.1),
    A.CLAHE(clip_limit=2.0, p=0.1),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.15, p=0.2),
]

model = YOLO("yolov8x-seg.pt")
model.train(
    data="../../datasets/operculum/data.yaml",
    epochs=100,
    patience=40,
    batch=4,
    imgsz=1080,               
    project="YOLOv8_Operculum",
    name="yolo8x_seg",
    optimizer="auto",
    weight_decay=0.0005,
    cos_lr=True,
    seed=0,
    rect=False,
    amp=True,
    cache="disk",              
    workers=2,
    plots=True,
    val=True,
    verbose=True,
    mask_ratio=4,            

    degrees=15.0,
    translate=0.1,
    scale=0.3,
    shear=0.0,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.0,

    hsv_h=0.01,
    hsv_s=0.4,
    hsv_v=0.3,
    bgr=0.0,

    mosaic=0.5,
    close_mosaic=15,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,

    augmentations=custom_transforms,
)

In [ ]:
model = YOLO(
    "runs/segment/YOLOv8_Operculum/yolo8x_seg/weights/best.pt"
)

metrics = model.val(
    data="../../datasets/operculum/data.yaml",
    task="segment"
)